# 3.0 - Entrenamiento de modelos

Torneo de modelos para predecir `SalePrice`:
- **Validación cruzada KFold (k = 10)** con optimización de hiperparámetros (`GridSearchCV`, al menos 3 configuraciones por modelo).
- **6 modelos:** regresiones (Ridge, Lasso, ElasticNet), ensambles (RandomForest, GradientBoosting) y una red neuronal (MLPRegressor).
- **Sin fuga de datos:** el pipeline de ingeniería de características (notebook 2.0) se ajusta **dentro de cada fold**, solo con los datos de entrenamiento de ese fold.
- Los modelos se entrenan con `SalePrice` en **dólares** (sin logaritmo). El torneo se decide en el **dataset de validación**, con el **RMSE real** (en dólares) y el **RMSE normalizado al rango**.
- **MLflow:** registra el torneo (un run padre `torneo_AAAAMMDD_HHMMSS` y un run hijo por modelo) y los 3 mejores modelos en el Model Registry como `house_pricing@champion`, `@challenger_1` y `@challenger_2`.

## 1. Configuración

Importa las librerías y las rutas del proyecto, y fija la semilla.

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
import json

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import yaml

from feature_engine.preprocessing import MatchVariables
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

from proyecto_pd.config import (
    FIGURES_DIR,
    MLFLOW_ARTIFACTS_DIR,
    MLFLOW_EXPERIMENT,
    MLFLOW_MODELO_REGISTRADO,
    MLFLOW_TRACKING_URI,
    MODELS_DIR,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    RAW_DATA_DIR,
)

pd.set_option("display.max_columns", 100)
plt.style.use("ggplot")

SEMILLA = 42

### 1.1 Configuración de MLflow

Conecta MLflow con el almacenamiento local del proyecto:
- **Tracking y Model Registry:** base de datos SQLite `mlflow.db`, en la raíz del proyecto.
- **Artefactos** (modelos, gráficas y tablas): carpeta `mlruns/`.
- **Experimento:** `house_pricing`. Se crea la primera vez.

Para explorar los resultados, ejecuta desde la raíz del proyecto `mlflow ui --backend-store-uri sqlite:///mlflow.db` y abre http://127.0.0.1:5000.

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
if mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT) is None:
    mlflow.create_experiment(MLFLOW_EXPERIMENT, artifact_location=MLFLOW_ARTIFACTS_DIR.as_uri())
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento: {MLFLOW_EXPERIMENT}")

## 2. Carga de datos

Para que el pipeline de features pueda ajustarse dentro de cada fold, los modelos reciben las **features crudas**. Se recrea exactamente la misma división 80/20 del notebook 2.0 (`random_state=42`) y se verifica que los `Id` coincidan con `validacion_fe.csv`.

El target de validación se toma de `validacion_fe.csv`. Se usa la columna `SalePrice` original, en dólares, no la versión en logaritmo.

In [ ]:
with open(PROJ_ROOT / "pipeline_config.yaml", encoding="utf-8") as f:
    MODEL_FEATURES = yaml.safe_load(f)["MODEL_FEATURES"]

train = pd.read_csv(RAW_DATA_DIR / "train.csv", index_col="Id")
X_train, X_val, y_train, _ = train_test_split(
    train[MODEL_FEATURES], train["SalePrice"], test_size=0.2, random_state=SEMILLA
)

validacion_fe = pd.read_csv(PROCESSED_DATA_DIR / "validacion_fe.csv", index_col="Id")
assert X_val.index.equals(validacion_fe.index), "El split no coincide con validacion_fe.csv"
y_val = validacion_fe["SalePrice"]

print(f"X_train: {X_train.shape} | X_val: {X_val.shape}")
print("Ids de validación coinciden con validacion_fe.csv: True")

## 3. Pipeline de ingeniería de características

Carga `models/pipeline_fe.joblib` (notebook 2.0) y lo **clona sin ajustar**: se conservan los mismos pasos y parámetros, pero sin los valores aprendidos. Así, en cada fold, la imputación, la codificación, el capping, las transformaciones y el escalado se aprenden solo con los datos de entrenamiento del fold.

In [ ]:
pipeline_fe = clone(joblib.load(MODELS_DIR / "pipeline_fe.joblib"))

[nombre for nombre, _ in pipeline_fe.steps]

## 4. Modelos y rejillas de hiperparámetros

Define los 6 modelos del torneo y las configuraciones a probar (todas con 3 o más). Para controlar el sobreajuste:
- Las regresiones usan regularización (`alpha`, `l1_ratio`). Como el target está en dólares, los `alpha` de Lasso y ElasticNet están en esa escala.
- RandomForest limita la profundidad, el tamaño mínimo de las hojas y las variables por división.
- GradientBoosting usa árboles poco profundos, tasa de aprendizaje y `subsample`.
- La red neuronal usa regularización L2 (`alpha`) y `early_stopping`, que reserva el 10% del fold de entrenamiento para detenerse.

In [ ]:
MODELOS = {
    "Ridge": (
        Ridge(),
        {"modelo__alpha": [0.1, 1, 10, 30, 100]},
    ),
    "Lasso": (
        Lasso(max_iter=20000),
        {"modelo__alpha": [10, 50, 100, 300, 1000]},
    ),
    "ElasticNet": (
        ElasticNet(max_iter=20000),
        {"modelo__alpha": [0.01, 0.1, 1], "modelo__l1_ratio": [0.2, 0.5, 0.8]},
    ),
    "RandomForest": (
        RandomForestRegressor(n_estimators=300, random_state=SEMILLA, n_jobs=-1),
        {
            "modelo__max_depth": [None, 15],
            "modelo__min_samples_leaf": [1, 3],
            "modelo__max_features": ["sqrt", 0.5],
        },
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(n_estimators=500, subsample=0.8, random_state=SEMILLA),
        {"modelo__learning_rate": [0.03, 0.1], "modelo__max_depth": [2, 3]},
    ),
    "MLP": (
        MLPRegressor(max_iter=2000, early_stopping=True, random_state=SEMILLA),
        {
            "modelo__hidden_layer_sizes": [(64,), (64, 32), (128, 64)],
            "modelo__alpha": [0.001, 0.01, 0.1],
        },
    ),
}

pd.DataFrame({
    "modelo": list(MODELOS),
    "configuraciones": [int(np.prod([len(v) for v in grid.values()])) for _, grid in MODELOS.values()],
})

## 5. Validación cruzada KFold (k = 10) con optimización de hiperparámetros

Para cada modelo, `GridSearchCV` evalúa todas las configuraciones con **KFold de 10 particiones**, usando el mismo `Pipeline(features + modelo)`. La métrica es el RMSE de `SalePrice` en dólares.

- Se guardan también los errores de entrenamiento (`return_train_score=True`) para medir la brecha entre train y CV, que indica sobreajuste.
- La mejor configuración se reentrena con todo `X_train`.
- **El dataset de validación no se usa en este paso.**

**MLflow:** todo el torneo queda en un run padre llamado `torneo_AAAAMMDD_HHMMSS` (fecha y hora de la ejecución), con los parámetros generales. Cada modelo tiene un run hijo con su rejilla, sus mejores hiperparámetros y sus métricas de CV: `rmse_cv`, `std_cv`, `rmse_train` y `brecha_cv_train`.

In [ ]:
kfold = KFold(n_splits=10, shuffle=True, random_state=SEMILLA)

busquedas = {}
runs_modelos = {}
with mlflow.start_run(run_name=f"torneo_{datetime.now():%Y%m%d_%H%M%S}") as run_torneo:
    mlflow.set_tag("tipo_run", "torneo")
    mlflow.log_params({
        "cv_folds": kfold.n_splits,
        "semilla": SEMILLA,
        "target": "SalePrice (USD)",
        "metrica_cv": "RMSE",
        "filas_train": len(X_train),
        "filas_validacion": len(X_val),
        "n_features": len(MODEL_FEATURES),
        "modelos": ", ".join(MODELOS),
    })

    for nombre, (modelo, grid) in MODELOS.items():
        with mlflow.start_run(run_name=nombre, nested=True) as run_modelo:
            pipeline = Pipeline([("features", clone(pipeline_fe)), ("modelo", modelo)])
            busqueda = GridSearchCV(
                pipeline,
                param_grid=grid,
                cv=kfold,
                scoring="neg_root_mean_squared_error",
                return_train_score=True,
                n_jobs=-1,
            )
            busqueda.fit(X_train, y_train)
            busquedas[nombre] = busqueda
            runs_modelos[nombre] = run_modelo.info.run_id

            res = busqueda.cv_results_
            i = busqueda.best_index_
            mlflow.set_tags({"tipo_run": "modelo", "modelo": nombre})
            mlflow.log_params({
                "rejilla": json.dumps(grid),
                "configuraciones_probadas": len(res["params"]),
                **{param.removeprefix("modelo__"): valor for param, valor in busqueda.best_params_.items()},
            })
            mlflow.log_metrics({
                "rmse_cv": -res["mean_test_score"][i],
                "std_cv": res["std_test_score"][i],
                "rmse_train": -res["mean_train_score"][i],
                "brecha_cv_train": -res["mean_test_score"][i] + res["mean_train_score"][i],
            })
        print(f"{nombre}: RMSE CV = {-busqueda.best_score_:,.0f} | {busqueda.best_params_}")

ID_RUN_TORNEO = run_torneo.info.run_id
print(f"\nRun del torneo en MLflow: {run_torneo.info.run_name} ({ID_RUN_TORNEO})")

### 5.1 Resultados de la validación cruzada

Para la mejor configuración de cada modelo muestra el RMSE (en dólares) promedio en train y en CV, su desviación entre los 10 folds y la **brecha CV − train**. Una brecha grande indica sobreajuste.

In [ ]:
filas = []
for nombre, busqueda in busquedas.items():
    res = busqueda.cv_results_
    i = busqueda.best_index_
    filas.append({
        "modelo": nombre,
        "rmse_train": -res["mean_train_score"][i],
        "rmse_cv": -res["mean_test_score"][i],
        "std_cv": res["std_test_score"][i],
        "brecha_cv_train": -res["mean_test_score"][i] + res["mean_train_score"][i],
        "configuraciones_probadas": len(res["params"]),
        "mejores_parametros": busqueda.best_params_,
    })

resultados_cv = pd.DataFrame(filas).sort_values("rmse_cv").reset_index(drop=True)
resultados_cv.round(0)

## 6. Torneo en el dataset de validación

Cada modelo (mejor configuración, entrenado con todo `X_train`) predice el dataset de validación directamente en dólares:
1. **RMSE real:** RMSE en dólares contra el `SalePrice` original.
2. **RMSE normalizado al rango (NRMSE):** RMSE real / (máximo − mínimo de `SalePrice` en validación). Se expresa como fracción del rango de precios.

Gana el modelo con el menor RMSE real en validación.

In [ ]:
rango_val = y_val.max() - y_val.min()

predicciones = {}
filas = []
for nombre, busqueda in busquedas.items():
    pred = busqueda.predict(X_val)
    predicciones[nombre] = pred
    rmse_real = root_mean_squared_error(y_val, pred)
    filas.append({
        "modelo": nombre,
        "rmse_real": rmse_real,
        "nrmse_rango": rmse_real / rango_val,
    })

torneo = pd.DataFrame(filas).sort_values("rmse_real").reset_index(drop=True)
print(f"Rango de SalePrice en validación: {y_val.min():,.0f} - {y_val.max():,.0f} (= {rango_val:,.0f})")
torneo.round({"rmse_real": 0, "nrmse_rango": 4})

## 7. Gráficas de desempeño por modelo

Para cada modelo, sobre el dataset de validación y en dólares:
- **Izquierda, real vs predicho:** cuanto más cerca estén los puntos de la diagonal, mejor.
- **Derecha, residuos (real − predicho) vs predicho:** idealmente aparecen centrados en 0 y sin patrón. Un abanico o una curva indica errores sistemáticos.

In [ ]:
fig, axes = plt.subplots(len(predicciones), 2, figsize=(12, 4.5 * len(predicciones)))
limites = [min(y_val.min(), min(p.min() for p in predicciones.values())),
           max(y_val.max(), max(p.max() for p in predicciones.values()))]

for (ax_real, ax_res), nombre in zip(axes, torneo["modelo"]):
    pred = predicciones[nombre]
    fila = torneo.set_index("modelo").loc[nombre]
    titulo = f"{nombre} | RMSE = {fila['rmse_real']:,.0f} | NRMSE = {fila['nrmse_rango']:.4f}"

    ax_real.scatter(y_val, pred, alpha=0.5, s=12)
    ax_real.plot(limites, limites, color="black", linestyle="--", linewidth=1)
    ax_real.set_xlabel("SalePrice real")
    ax_real.set_ylabel("SalePrice predicho")
    ax_real.set_title(titulo, fontsize=10)

    ax_res.scatter(pred, y_val - pred, alpha=0.5, s=12)
    ax_res.axhline(0, color="black", linestyle="--", linewidth=1)
    ax_res.set_xlabel("SalePrice predicho")
    ax_res.set_ylabel("Residuo (real - predicho)")
    ax_res.set_title(f"{nombre} - residuos", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "modelos_real_vs_predicho.png")
plt.show()

## 8. Comparación del torneo

Compara el **RMSE real** y el **RMSE normalizado al rango** de todos los modelos en validación, ordenados de mejor a peor.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(torneo["modelo"], torneo["rmse_real"])
axes[0].set_xlabel("RMSE real (USD)")
axes[0].set_title("RMSE real en validación")
for i, v in enumerate(torneo["rmse_real"]):
    axes[0].text(v, i, f" {v:,.0f}", va="center")

axes[1].barh(torneo["modelo"], torneo["nrmse_rango"], color="tab:blue")
axes[1].set_xlabel("RMSE / rango")
axes[1].set_title("RMSE normalizado al rango en validación")
for i, v in enumerate(torneo["nrmse_rango"]):
    axes[1].text(v, i, f" {v:.4f}", va="center")

for ax in axes:
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "modelos_torneo.png")
plt.show()

ganador = torneo.iloc[0]
print(f"Mejor modelo en validación: {ganador['modelo']} "
      f"(RMSE real = {ganador['rmse_real']:,.0f} | NRMSE = {ganador['nrmse_rango']:.4f})")

## 9. Registro del torneo en MLflow

Completa en MLflow los resultados de la validación:
- **Run de cada modelo:** métricas `rmse_validacion`, `nrmse_validacion` y `posicion_torneo`, más su gráfica de real vs predicho y residuos.
- **Run padre del torneo:** las tablas de CV y del torneo en CSV, las dos gráficas comparativas y las etiquetas del modelo ganador.

In [ ]:
for posicion, fila in enumerate(torneo.itertuples(), start=1):
    pred = predicciones[fila.modelo]
    fig, (ax_real, ax_res) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax_real.scatter(y_val, pred, alpha=0.5, s=12)
    ax_real.plot(limites, limites, color="black", linestyle="--", linewidth=1)
    ax_real.set_xlabel("SalePrice real")
    ax_real.set_ylabel("SalePrice predicho")
    ax_real.set_title(f"{fila.modelo} | RMSE = {fila.rmse_real:,.0f} | NRMSE = {fila.nrmse_rango:.4f}", fontsize=10)
    ax_res.scatter(pred, y_val - pred, alpha=0.5, s=12)
    ax_res.axhline(0, color="black", linestyle="--", linewidth=1)
    ax_res.set_xlabel("SalePrice predicho")
    ax_res.set_ylabel("Residuo (real - predicho)")
    ax_res.set_title(f"{fila.modelo} - residuos", fontsize=10)
    plt.tight_layout()

    with mlflow.start_run(run_id=runs_modelos[fila.modelo]):
        mlflow.log_metrics({
            "rmse_validacion": fila.rmse_real,
            "nrmse_validacion": fila.nrmse_rango,
            "posicion_torneo": posicion,
        })
        mlflow.log_figure(fig, "real_vs_predicho.png")
    plt.close(fig)

with mlflow.start_run(run_id=ID_RUN_TORNEO):
    mlflow.log_text(resultados_cv.to_csv(index=False), "resultados_cv.csv")
    mlflow.log_text(torneo.to_csv(index=False), "torneo_validacion.csv")
    mlflow.log_artifact(FIGURES_DIR / "modelos_real_vs_predicho.png")
    mlflow.log_artifact(FIGURES_DIR / "modelos_torneo.png")
    mlflow.log_metrics({"rmse_validacion_ganador": torneo.iloc[0]["rmse_real"],
                        "nrmse_validacion_ganador": torneo.iloc[0]["nrmse_rango"]})
    mlflow.set_tag("ganador", torneo.iloc[0]["modelo"])

print(f"Torneo registrado en MLflow (run {ID_RUN_TORNEO}) con {len(runs_modelos)} runs de modelos.")

## 10. Guardar el modelo ganador

Toma el modelo ganador del torneo con sus mejores hiperparámetros y lo **reentrena con train + validación** (1460 registros), para aprovechar todos los datos etiquetados al predecir test.

- Se guarda el **pipeline completo**: ingeniería de características (notebook 2.0) + modelo. Ambos se reajustan juntos con los datos combinados, sin fuga.
- Su desempeño esperado es el medido en el torneo, porque este modelo exacto ya no tiene un conjunto independiente para evaluarlo.

Se guarda en `models/modelo_ganador.joblib`.

In [ ]:
nombre_ganador = torneo.iloc[0]["modelo"]

X_completo = pd.concat([X_train, X_val])
y_completo = pd.concat([y_train, y_val])

modelo_ganador = clone(busquedas[nombre_ganador].best_estimator_)
modelo_ganador.fit(X_completo, y_completo)

joblib.dump(modelo_ganador, MODELS_DIR / "modelo_ganador.joblib")

print(f"Modelo ganador: {nombre_ganador} | {busquedas[nombre_ganador].best_params_}")
print(f"Entrenado con: {X_completo.shape[0]} registros (train + validación)")
print(f"Guardado en: {MODELS_DIR / 'modelo_ganador.joblib'}")

modelo_ganador

## 11. Registro en el Model Registry: champion y challengers

Registra en MLflow los **3 mejores modelos del torneo** como versiones del modelo `house_pricing`:

| Posición | Alias |
|---|---|
| 1 | `@champion` |
| 2 | `@challenger_1` |
| 3 | `@challenger_2` |

Para cada uno:
1. **Reentrenamiento:** se reentrena con sus mejores hiperparámetros sobre **train + validación** (1460 registros). El champion es el mismo `modelo_ganador` de la sección 10.
2. **Pipeline completo de inferencia:** selección de features (`MatchVariables`) → ingeniería de características → modelo. Recibe datos crudos y devuelve `SalePrice` en dólares.
3. **Registro:** se guarda en el run del modelo, con su **firma** (columnas y tipos de entrada y salida) y un **ejemplo de entrada**. Se usa el formato `cloudpickle`, porque el formato por defecto (skops) no puede guardar objetos de feature-engine. Igual que con los `.joblib`, solo se deben cargar modelos de origen confiable.
4. **Alias y etiquetas:** `tipo_modelo` (p. ej. `gradiente_boosting`), `familia` (`regresion`, `ensamble` o `red_neuronal`), `posicion_torneo`, métricas de validación, hiperparámetros y datos de entrenamiento.

Cada ejecución crea versiones nuevas y **mueve los alias** a ellas. Las versiones anteriores quedan en el historial del registro.

In [ ]:
TIPOS_MODELO = {
    "Ridge": ("ridge", "regresion"),
    "Lasso": ("lasso", "regresion"),
    "ElasticNet": ("elastic_net", "regresion"),
    "RandomForest": ("random_forest", "ensamble"),
    "GradientBoosting": ("gradiente_boosting", "ensamble"),
    "MLP": ("red_neuronal_mlp", "red_neuronal"),
}
ALIAS = ["champion", "challenger_1", "challenger_2"]

cliente = MlflowClient()
seleccion_features = MatchVariables(missing_values="ignore", verbose=False).fit(X_completo)
columnas_numericas = X_completo.select_dtypes(include="number").columns
X_firma = X_completo.astype({col: "float64" for col in columnas_numericas})

registrados = []
for posicion, (alias, fila) in enumerate(zip(ALIAS, torneo.head(len(ALIAS)).itertuples()), start=1):
    nombre = fila.modelo
    if nombre == nombre_ganador:
        modelo_final = modelo_ganador
    else:
        modelo_final = clone(busquedas[nombre].best_estimator_).fit(X_completo, y_completo)
    pipeline_registro = Pipeline([("seleccionar_features", seleccion_features)] + modelo_final.steps)
    firma = infer_signature(X_firma, pipeline_registro.predict(X_firma))

    with mlflow.start_run(run_id=runs_modelos[nombre]):
        info = mlflow.sklearn.log_model(
            pipeline_registro,
            name="modelo",
            signature=firma,
            input_example=X_firma.head(5),
            registered_model_name=MLFLOW_MODELO_REGISTRADO,
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )
        mlflow.set_tag("alias_registro", alias)

    version = info.registered_model_version
    cliente.set_registered_model_alias(MLFLOW_MODELO_REGISTRADO, alias, version)
    tipo_modelo, familia = TIPOS_MODELO[nombre]
    etiquetas = {
        "tipo_modelo": tipo_modelo,
        "familia": familia,
        "modelo": nombre,
        "posicion_torneo": posicion,
        "rmse_validacion": round(fila.rmse_real),
        "nrmse_validacion": round(fila.nrmse_rango, 4),
        "hiperparametros": json.dumps(
            {k.removeprefix("modelo__"): v for k, v in busquedas[nombre].best_params_.items()}
        ),
        "entrenado_con": f"train + validacion ({len(X_completo)} filas)",
        "target": "SalePrice (USD)",
    }
    for clave, valor in etiquetas.items():
        cliente.set_model_version_tag(MLFLOW_MODELO_REGISTRADO, version, clave, str(valor))

    registrados.append({"alias": alias, "version": version, **etiquetas})

with mlflow.start_run(run_id=ID_RUN_TORNEO):
    mlflow.set_tags({f"registro_{r['alias']}": f"{r['modelo']} (v{r['version']})" for r in registrados})

pd.DataFrame(registrados)[["alias", "version", "modelo", "tipo_modelo", "familia", "rmse_validacion", "nrmse_validacion"]]

### 11.1 Verificación del registro

Carga cada alias desde el Model Registry y comprueba dos cosas:
- El alias apunta a la versión recién registrada.
- El champion cargado desde MLflow da las mismas predicciones que `modelo_ganador` sobre los datos de validación.

In [ ]:
for registro in registrados:
    version = cliente.get_model_version_by_alias(MLFLOW_MODELO_REGISTRADO, registro["alias"])
    print(f"@{registro['alias']}: versión {version.version} | {version.tags['modelo']} "
          f"({version.tags['tipo_modelo']}, {version.tags['familia']}) | "
          f"RMSE validación = {int(version.tags['rmse_validacion']):,}")

champion = mlflow.sklearn.load_model(f"models:/{MLFLOW_MODELO_REGISTRADO}@champion")
print(f"\nChampion de MLflow = modelo_ganador: {np.allclose(champion.predict(X_val), modelo_ganador.predict(X_val))}")